# 04 · Sessioni pulite

Le sessioni senza guasto sono il training del VAE: devono essere stazionarie dentro la sessione e confrontabili tra sessioni. Qui: deriva dall'inizio alla fine allo stesso carico, memoria di database e cache nel tempo, differenze tra sessioni, cadenza dei campioni, correlazioni tra feature.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90)

In [ ]:
DATA = Path("/home/tecnico/datasets/pilot")
STEP = 30

clean = {k: r for k, r in ((k, ex.load(a, STEP)) for k, a in ex.find_runs(DATA).items()) if r.kind == "clean"}
for name, run in clean.items():
    levels = sorted(int(r) for r in set(run.rate[run.rate >= 0]))
    print(f"{name}: split {run.labels.get('split')}, {run.t_rel[-1] / 60:.0f} min osservati, livelli {levels}")

## Deriva dentro la sessione

Per ogni livello (rate, mix) presente sia nella prima sia nella seconda metà dell'osservazione: log2 del rapporto tra la mediana della seconda metà e quella della prima, poi la mediana sui livelli. 0 = stabile, 1 = raddoppiato, −1 = dimezzato.

In [ ]:
def drift(run):
    steady = ex.steady_mask(run) & (run.t_rel > 0)
    late = run.t_rel > np.median(run.t_rel[steady]) if steady.any() else steady
    out = {}
    for feature in run.features:
        first = ex.by_level(run, feature, steady & ~late)
        second = ex.by_level(run, feature, steady & late)
        common = first.index.intersection(second.index)
        out[feature] = np.log2(second.loc[common] / first.loc[common]).replace([np.inf, -np.inf], np.nan).median()
    return pd.DataFrame(out)


for name, run in clean.items():
    d = drift(run)
    fig, a = plt.subplots(figsize=(12, 0.3 * len(d) + 1.5))
    image = a.imshow(d.to_numpy(float), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    a.set_xticks(range(len(d.columns)), d.columns, rotation=45, ha="right")
    a.set_yticks(range(len(d.index)), d.index, fontsize=8)
    a.set_title(f"{name}: log2(seconda metà / prima metà)")
    fig.colorbar(image)
    plt.show()

## Memoria di database e cache nel tempo

I database accumulano dati per tutta la sessione (post, prenotazioni): una crescita qui è attesa, ma il modello deve vederla nel training.

In [ ]:
for name, run in clean.items():
    stores = [s for s in run.services if any(k in s for k in ("mongo", "redis", "memcached"))]
    fig, a = plt.subplots(figsize=(13, 4))
    for service in stores:
        a.plot(run.t_rel / 60, ex.series(run, service, "mem_working_set") / 2**20, label=service)
    a.set(title=name, xlabel="minuti dall'iniezione", ylabel="MiB")
    a.legend(fontsize=7, ncol=3)
    plt.show()

## Confronto tra sessioni

Mediana a carico stabile di CPU (mC) e memoria (MiB) per servizio, a un livello di rate comune a tutte le sessioni, e coefficiente di variazione tra sessioni. Le sessioni con livelli mai visti non hanno livelli in comune con le altre.

In [ ]:
LEVEL = None  # req/s del confronto; None prende il livello centrale tra quelli comuni
if len(clean) < 2:
    print("Servono almeno due sessioni pulite in", DATA)
else:
    for feature, scale, unit in (("cpu", 1000, "mC"), ("mem_working_set", 2**-20, "MiB")):
        per_session = {n: ex.by_level(r, feature).groupby(level="rate").median() * scale for n, r in clean.items()}
        common = sorted(set.intersection(*(set(v.index) for v in per_session.values())))
        if not common:
            print(f"{feature}: nessun livello di rate comune a tutte le sessioni")
            continue
        level = LEVEL or common[len(common) // 2]
        compare = pd.DataFrame({n: v.loc[level] for n, v in per_session.items()})
        compare["CV"] = compare.std(axis=1) / compare.mean(axis=1)
        print(f"{feature} ({unit}) a {level} req/s")
        display(compare.round(2).sort_values("CV", ascending=False))

## Cadenza dei campioni

Intervalli tra campioni consecutivi di cAdvisor (CPU dei container applicativi) letti dai dati grezzi, e quota di celle vuote della griglia con passi diversi.

In [ ]:
import gzip
import json

for name, run in clean.items():
    with gzip.open(run.attempt / "prometheus" / "container_cpu_usage_seconds_total.json.gz", "rt") as f:
        raw = json.load(f)
    gaps = np.concatenate([np.diff([float(t) for t, _ in s["values"]]) for s in raw
                           if s["metric"].get("container") not in ("", "POD", None) and len(s["values"]) > 1])
    empty = {}
    for step in (15, 30, 60):
        r = ex.load(run.attempt, step)
        after = r.t_rel > 0
        present = r.mask[after].any(axis=0)
        empty[f"{step} s"] = round(float(1 - r.mask[after][:, present].mean()), 3)
    print(f"{name}: intervallo mediano {np.median(gaps):.1f} s, p95 {np.percentile(gaps, 95):.1f} s, massimo {gaps.max():.1f} s; celle vuote {empty}")
    plt.figure(figsize=(8, 2.5))
    plt.hist(gaps, bins=np.arange(0, 61, 1))
    plt.xlabel("secondi tra due campioni")
    plt.title(name)
    plt.show()

## Correlazioni tra feature

Correlazione di Spearman tra le feature a carico stabile, mediata sui servizi che lavorano (CPU mediana sopra 5 mC). Feature quasi ridondanti (|ρ| vicino a 1) si possono accorpare.

In [ ]:
for name, run in clean.items():
    steady = ex.steady_mask(run) & (run.t_rel > 0)
    cpu = run.features.index("cpu")
    busy = [i for i in range(len(run.services)) if np.nanmedian(run.x[steady, i, cpu]) > 0.005]
    if not busy:
        continue
    mats = [pd.DataFrame(run.x[steady, i, :], columns=run.features).rank().corr() for i in busy]
    mean = pd.concat(mats).groupby(level=0).mean().loc[run.features, run.features]
    fig, a = plt.subplots(figsize=(8, 7))
    image = a.imshow(mean.to_numpy(float), cmap="RdBu_r", vmin=-1, vmax=1)
    a.set_xticks(range(len(run.features)), run.features, rotation=90)
    a.set_yticks(range(len(run.features)), run.features)
    a.set_title(f"{name}: Spearman medio su {len(busy)} servizi")
    fig.colorbar(image)
    plt.show()